# 01 · Data Collection

**IPO Listing Gain Predictor** — predicting how much an Indian mainboard IPO (2022–2026) gains or
loses on its **listing day**, using only what is known *before* it lists: the grey market premium
(GMP), the subscription numbers, and the mood of the wider market (Nifty 50).

This first notebook just **loads the raw files and stacks them into clean tables**. No cleaning or
modelling yet — that comes in notebooks 02–05.

I have three kinds of raw data:

1. **`ipo_data.csv`** — one row per IPO (issue price, subscription, GMP for 5 days, fundamentals,
   and the listing-day result I want to predict).
2. **`ipo22.csv … ipo26.csv`** — year-wise performance sheets. I only need the **listing date** from
   these, which the main file does not have.
3. **`nif22.csv … nif26.csv`** — daily Nifty 50 history, used later to measure market mood around
   each listing.


In [1]:
import pandas as pd

RAW = "../data/raw"
PROC = "../data/processed"
target = "listing_gain_close_pct"   # the column we will predict everywhere


## 1. The main IPO file

In [2]:
ipo = pd.read_csv(f"{RAW}/ipo_data.csv")
print("shape:", ipo.shape)
ipo.head()


shape: (312, 39)


,year,company,issue_price,listing_price,closing_price,listing_gain_open_pct,listing_gain_close_pct,day_change_after_pct,retail_quota_pct,total_sub_x,...,ai_pred_pct,actual_gain_pct,slug,url,gmp_day2_rs,gmp_day3_rs,gmp_day4_rs,gmp_day4_pct,gmp_day5_rs,gmp_day5_pct
0,2022,Abans HoldingsIPO,270.0,216.05,258.00,-19.98,-4.44,15.54,35.0,80.80,...,NaN,-20.0,abans-holdings,https://ipogyani.com/listed-ipo/2022/abans-hol...,17.0,12.0,15.0,5.56,5.0,1.85
1,2022,Adani WilmarIPO,230.0,268.25,268.25,16.63,16.63,0.00,35.0,182.44,...,NaN,16.6,adani-wilmar,https://ipogyani.com/listed-ipo/2022/adani-wilmar,60.0,50.0,45.0,19.57,35.0,15.22
2,2022,Aether IndustriesIPO,642.0,774.38,774.40,20.62,20.62,0.00,10.0,148.75,...,NaN,20.6,aether-industries,https://ipogyani.com/listed-ipo/2022/aether-in...,5.0,5.0,10.0,1.56,0.0,0.00
3,2022,AGS Transact TechnologiesIPO,175.0,161.05,154.00,-7.97,-12.00,-4.03,35.0,227.51,...,NaN,-8.0,ags-transact-technologies,https://ipogyani.com/listed-ipo/2022/ags-trans...,20.0,30.0,20.0,11.43,20.0,11.43
4,2022,Archean Chemical IndustriesIPO,407.0,458.16,493.00,12.57,21.13,8.56,35.0,16.69,...,NaN,12.6,archean-chemical-industries,https://ipogyani.com/listed-ipo/2022/archean-c...,60.0,60.0,70.0,17.20,75.0,18.43


In [3]:
# how many IPOs per year?
ipo["year"].value_counts().sort_index()


year
2022     37
2023     60
2024     89
2025    106
2026     20
Name: count, dtype: int64

## 2. Listing dates (stack the 5 year files)

The year files all have the same columns, so I read them in a loop and `concat` them into one table.
I only keep the company name and the listing date.


In [4]:
years = ["22", "23", "24", "25", "26"]

listings = pd.concat([pd.read_csv(f"{RAW}/ipo{y}.csv") for y in years], ignore_index=True)
listings.columns = [c.strip() for c in listings.columns]   # the headers have stray spaces

# "Listed On" looks like 'Fri, Dec 30, 2022' -> turn it into a real date
listings["list_date"] = pd.to_datetime(listings["Listed On"].str.strip(),
                                       format="%a, %b %d, %Y", errors="coerce")
listings = listings.dropna(subset=["list_date"]).drop_duplicates("Company Name")

print("listing records:", len(listings))
listings[["Company Name", "list_date"]].head()


listing records: 325


,Company Name,list_date
0,Elin Electronics Ltd.,2022-12-30
1,KFin Technologies Ltd.,2022-12-29
2,Abans Holdings Ltd.,2022-12-23
3,Landmark Cars Ltd.,2022-12-23
4,Sula Vineyards Ltd.,2022-12-22


## 3. Nifty 50 daily history (stack the 5 files)

Same idea — read each year's file and stack them. I keep only the date and the closing level, then
sort by date so it is ready for the market-mood features later.


In [5]:
nifty = pd.concat([pd.read_csv(f"{RAW}/nif{y}.csv") for y in years], ignore_index=True)
nifty.columns = [c.strip() for c in nifty.columns]

nifty["Date"]  = pd.to_datetime(nifty["Date"].str.strip(), format="%d-%b-%Y", errors="coerce")
nifty["Close"] = pd.to_numeric(nifty["Close"], errors="coerce")
nifty = (nifty.dropna(subset=["Date", "Close"])
              .drop_duplicates("Date")
              .sort_values("Date")
              .reset_index(drop=True))

print("nifty days:", len(nifty), "|", nifty["Date"].min().date(), "->", nifty["Date"].max().date())
nifty[["Date", "Close"]].tail()


nifty days: 1097 | 2022-01-03 -> 2026-06-09


,Date,Close
1092,2026-06-03,23405.60
1093,2026-06-04,23416.55
1094,2026-06-05,23366.70
1095,2026-06-08,23123.00
1096,2026-06-09,23242.10


## 4. Save the combined tables

I write the three tidy tables to `data/processed/` so the next notebooks can just read them instead
of re-stacking the raw files every time.


In [6]:
ipo.to_csv(f"{PROC}/ipo_main.csv", index=False)
listings[["Company Name", "list_date"]].to_csv(f"{PROC}/listings.csv", index=False)
nifty[["Date", "Close"]].to_csv(f"{PROC}/nifty.csv", index=False)
print("saved ipo_main.csv, listings.csv, nifty.csv to data/processed/")


saved ipo_main.csv, listings.csv, nifty.csv to data/processed/


**Recap.** I now have three clean inputs: 312 IPOs, a date for most of them in the year files, and
~860 days of Nifty closes. Next: explore the data and find what actually moves listing gains.
